**Agentic RAG Implementation**

This document shows:

* How tools are actually attached to an LLM system
* How API keys are provided
* How vector DBs are connected
* How browser tools are attached
* How code execution tools are exposed
* How memory is attached
* How planning agents are implemented
* How MCP servers are configured
* How the orchestrator exposes tools to the LLM

This is NOT a production-ready system.

The goal is to help you understand the architecture of a real Agentic RAG system.

#Overview

If a normal RAG can work by searching for context through a vector database guided by LLM Prompts, then an agentic RAG can work by defining some tools at disposal of the model and Prompting an LLM on what tool to use when based on a defined workflow.

Now these tools can be provided by the LLM in the form of -
1. Retrieval Tools
2. Wrapper Libraries
3. MCP Servers
4. API Tools
5. Database Tools
6. Code Execution Tools
7. Browser/Web Navigation Tools
8. Memory Tools  
9. Planning/Reasoning Tools
10. Multi Agent Tools
11. Knowledge Graph Tools
12. Local System Tools
13. Communication Tools

**NOTE :**

The LLM itself:

* **does NOT** execute code
* **does NOT** access databases
* **does NOT** open browsers
* **does NOT** access filesystem directly

RUNTIME USER IS THE EXECUTOR.

In general, a tool is exposed to LLM in a JSON format with -
1. Tool Name
2. Tool Description
3. Input Schema
4. Execution Function

| Tool Type                    | How They Are Usually Provided                                                                                   |
| ---------------------------- | --------------------------------------------------------------------------------------------------------------- |
| Retrieval Tools              | Connected through vector DB SDKs, search APIs, retriever wrappers, or MCP servers                              |
| Wrapper Libraries            | Custom Python/JS functions wrapped into tool schemas/framework abstractions (custom made)                                     |
| MCP Servers                  | Registered through config files, startup configs, CLI args, or runtime connectors (written in .config files)                              |
| API Tools                    | Connected using API keys, endpoints, OAuth tokens, `.env` configs                                               |
| Database Tools               | DB connection strings, credentials, ORM connectors, query wrappers                                              |
| Code Execution Tools         | Sandboxes, Docker runtimes, Jupyter kernels, Kubernetes executors, remote execution SDKs, MCP execution servers |
| Browser/Web Navigation Tools | Browser automation libraries/services like Playwright, Selenium, Puppeteer, Browserbase                         |
| Memory Tools                 | Vector DBs, Redis, session stores, long-term memory DBs, persistent storage layers                              |
| Planning/Reasoning Tools     | Mostly prompt engineering, workflow graphs, planner-executor architectures, orchestration logic                 |
| Multi-Agent Tools            | Agent orchestration frameworks, message buses, shared memory systems, workflow coordinators                     |
| Knowledge Graph Tools        | Graph DB connectors, Cypher/SPARQL query engines, ontology APIs                                                 |
| Local System Tools           | OS wrappers, filesystem permissions, shell executors, local MCP servers                                         |
| Communication Tools          | SDKs/APIs for Slack, email, Discord, Teams, Twilio, WhatsApp, webhooks                                          |

**Model Workflow**

User Query -> LLM analyzes question -> LLM decides tool needed -> Framework converts decision into tool call -> Python function executes -> Results returned to LLM -> LLM reasons again -> Final answer generated

#1. Install Libraries

In [ ]:
!pip install openai langchain langgraph chromadb sentence-transformers \
    selenium beautifulsoup4 playwright redis neo4j \
    duckduckgo-search pypdf tiktoken

In [ ]:
from openai import OpenAI
from langchain.tools import tool
from langchain_openai import ChatOpenAI

import chromadb
import redis
from neo4j import GraphDatabase

from duckduckgo_search import DDGS
from selenium import webdriver
from selenium.webdriver.common.by import By

import subprocess
import os
import json

#.dotenv File

In [ ]:
import os

# LLM
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"

# Vector DB
os.environ["CHROMA_DB_PATH"] = "./chroma_db"

# Redis Memory
os.environ["REDIS_URL"] = "redis://localhost:6379"

# Neo4j Graph DB
os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "password"

# Browser Tool
os.environ["PLAYWRIGHT_BROWSERS_PATH"] = "./playwright"

# Example External API
os.environ["SERP_API_KEY"] = "YOUR_SERP_API_KEY"

#.config file (MCP Server)

In [ ]:
#JSON FILE
 {
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": [
        "@modelcontextprotocol/server-filesystem",
        "/content/project"
      ]
    }
  }
}

#2. Initialize LLM

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

#RETRIVAL TOOL

In [ ]:
chroma_client = chromadb.PersistentClient(
    path=os.environ["CHROMA_DB_PATH"]
)

collection = chroma_client.get_or_create_collection(
    name="documents"
)

In [ ]:
@tool
def retrieve_docs(query: str):
    """Search vector database for relevant documents."""

    results = collection.query(
        query_texts=[query],
        n_results=5
    )

    return results

#WEB SEARCH TOOL (WRAPPER)

In [ ]:
@tool
def web_search(query: str):
    """Search the internet for recent information."""

    results = []

    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):
            results.append(r)

    return results

#BROWSER AUTOMATION TOOL

In [ ]:
def open_website(url: str):
    """Open a website and extract text."""

    driver = webdriver.Chrome()

    driver.get(url)

    body = driver.find_element(By.TAG_NAME, "body")

    text = body.text[:5000]

    driver.quit()

    return text

#CODE EXECUTION TOOL

In [ ]:
@tool
def execute_python(code: str):
    """Execute Python code safely."""

    with open("temp_script.py", "w") as f:
        f.write(code)

    result = subprocess.run(
        ["python", "temp_script.py"],
        capture_output=True,
        text=True,
        timeout=20
    )

    return {
        "stdout": result.stdout,
        "stderr": result.stderr
    }

#MEMORY TOOL

In [ ]:
redis_client = redis.Redis.from_url(
    os.environ["REDIS_URL"]
)

In [ ]:
@tool
def store_memory(key: str, value: str):
    """Store information into memory."""

    redis_client.set(key, value)

    return "stored"

In [ ]:
@tool
def get_memory(key: str):
    """Retrieve memory."""

    value = redis_client.get(key)

    if value:
        return value.decode()

    return "not found"

#KNOWLEDGE GRAPH TOOL

In [ ]:
neo_driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(
        os.environ["NEO4J_USERNAME"],
        os.environ["NEO4J_PASSWORD"]
    )
)

In [ ]:
@tool
def query_graph(cypher_query: str):
    """Run Cypher queries on knowledge graph."""

    with neo_driver.session() as session:
        result = session.run(cypher_query)

        return [r.data() for r in result]

#LOCAL SYSTEM TOOL

In [ ]:
ALLOWED_COMMANDS = ["ls", "pwd", "cat"]

In [ ]:
@tool
def run_shell(command: str):
    """Run safe shell commands."""

    cmd = command.split()[0]

    if cmd not in ALLOWED_COMMANDS:
        return "Command not allowed"

    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True
    )

    return result.stdout

#COMMUNICATION TOOL

In [ ]:
@tool
def send_message(message: str):
    """Send notifications/messages."""

    print(f"MESSAGE SENT: {message}")

    return "message sent"

#PLANNING/REASONING TOOL

In [ ]:
SYSTEM_PROMPT = """
You are an advanced Agentic RAG system.

Available capabilities:

1. Retrieval from vector database
2. Web search
3. Browser navigation
4. Python code execution
5. Graph database reasoning
6. Memory storage
7. Shell access
8. Communication tools

Workflow Rules:

- First determine if retrieval is needed.
- Use vector retrieval for internal knowledge.
- Use web search for recent information.
- Use browser automation if webpage extraction needed.
- Use graph reasoning for relationship-heavy questions.
- Use code execution for calculations or data analysis.
- Store useful information into memory.
- Verify results before final answer.

Always explain reasoning step-by-step.
"""

#MULTI-AGENT TOOL

In [ ]:
research_agent_prompt = "Research and retrieve information"

coding_agent_prompt = "Execute and debug code"

verification_agent_prompt = "Verify factual correctness"

#Step 3: Tool Registration

In [ ]:
tools = [
    retrieve_docs,
    web_search,
    open_website,
    execute_python,
    store_memory,
    get_memory,
    query_graph,
    run_shell,
    send_message
]

In [ ]:
#JSON FILE
{
  "name": "retrieve_docs",
  "description": "Search vector database",
  "parameters": {
      ...
  }
}

#Step 4: Agent Creation

In [ ]:
from langchain.agents import initialize_agent
from langchain.agents import AgentType

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)

#Step 5: Running the Agent

In [ ]:
response = agent.invoke(
    """
    Compare NVIDIA and AMD AI revenues,
    search recent information,
    use graph reasoning if needed,
    and summarize findings.
    """
)